# 02 · Geospatial analysis
Coordinates, distance-to-amenity features and price comparison across locations. GeoPandas is used when installed.

In [ ]:
import sys, json
sys.path.insert(0, "..")          # make `src` importable from /notebooks
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config
pd.options.display.float_format = "{:,.2f}".format
from src.geospatial import add_distance_features, neighborhood_price_summary, to_geojson

In [ ]:
clean = pd.read_csv(config.CLEAN_CSV, parse_dates=["sale_date"])
amen = pd.read_csv(config.AMENITIES_CSV)
geo = add_distance_features(clean, amen)
geo["price_per_sqft"] = geo["sale_price"] / geo["living_area"]
geo[["dist_downtown_km","dist_park_km","dist_school_km","dist_transit_km"]].describe().T

## Where are the expensive areas?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(geo.longitude, geo.latitude, c=geo.price_per_sqft, s=6, cmap="RdYlBu_r", alpha=.7)
for kind, mk in {"downtown": "*", "park": "^", "school": "s", "transit": "D"}.items():
    a = amen[amen.amenity_type == kind]; ax.scatter(a.longitude, a.latitude, marker=mk, edgecolor="k", facecolor="none", s=60 if kind=="downtown" else 25, label=kind)
fig.colorbar(sc, label="$/sqft"); ax.legend(); ax.set_xlabel("longitude"); ax.set_ylabel("latitude"); plt.show()

## Do distances explain price?

In [ ]:
geo[["price_per_sqft","dist_downtown_km","dist_park_km","dist_school_km","dist_transit_km"]].corr(method="spearman")["price_per_sqft"].round(3)

In [ ]:
neighborhood_price_summary(clean)

## GeoPandas / GeoJSON export (GeoPandas optional)

In [ ]:
path = to_geojson(clean.sample(500, random_state=0), config.DATA_PROCESSED / "sample_points.geojson")
print("wrote", path)
try:
    import geopandas as gpd
    gdf = gpd.read_file(path); print(gdf.crs, len(gdf))
except ImportError:
    print("geopandas not installed - GeoJSON written with the plain-JSON fallback")